# Repump-on multilevel disk-sampling analysis

This notebook reads the fixed-speed multilevel sampling CSV and provides adjustable views of the same data used for the saved analysis figures. For the current repump-on CSV, remember that `untrapped_dark` is a legacy label: with the repumper enabled it means the trajectory visited `F=1` at least once, not that `F=1` was permanently terminal.

In [ ]:
%matplotlib inline

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'mot_multilevel':
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
elif PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

src = PROJECT_ROOT / 'src'
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

CSV_PATH = PROJECT_ROOT / 'outputs/statistics/mot_multilevel/sampling_repump_on_50_disks_25_points/multilevel_launch_samples.csv'
FIGURES_DIR = PROJECT_ROOT / 'outputs/figures/mot_multilevel/sampling_repump_on_50_disks_25_points_analysis'
CSV_PATH

PosixPath('/home/ajrosy/pMOT_MonteCarlo/outputs/statistics/mot_multilevel/sampling_repump_on_50_disks_25_points/multilevel_launch_samples.csv')

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pmot.mot_multilevel.sampling_analysis import run_sampling_csv_analysis

plt.style.use('default')
df = pd.read_csv(CSV_PATH)
df['lifetime_us'] = 1e6 * df['lifetime_s']
df['first_f1_visit_us'] = 1e6 * df['dark_entry_time_s']
df['s_mm'] = 1e3 * df['s_m']
df['minimum_radius_mm'] = 1e3 * df['minimum_radius_m']
df['final_radius_mm'] = 1e3 * df['final_radius_m']
df.head()

,disc_index,point_index,theta_rad,phi_rad,theta_prime_rad,s_m,radial_distance_m,launch_speed_m_per_s,initial_state_index,initial_f,...,yf_m,zf_m,vxf_m_per_s,vyf_m_per_s,vzf_m_per_s,lifetime_us,first_f1_visit_us,s_mm,minimum_radius_mm,final_radius_mm
0,0,0,0.880246,0.42378,0.000000,0.000000,0.015,8.0,3,2,...,0.002439,0.004314,-2.815043,-1.697782,-5.082504,615.760853,615.760853,0.000000,7.712623,7.712623
1,0,1,0.880246,0.42378,0.257444,0.000315,0.015,8.0,5,2,...,0.002670,0.004478,-3.090611,-2.057782,-4.945771,974.513477,NaN,0.314906,7.803654,7.803654
2,0,2,0.880246,0.42378,5.109928,0.003388,0.015,8.0,6,2,...,0.002644,0.005862,-4.220438,-2.023319,-6.046655,901.540010,901.540010,3.387703,7.038512,7.038512
3,0,3,0.880246,0.42378,3.811605,0.004047,0.015,8.0,4,2,...,-0.001431,0.005271,-3.109760,-1.531060,-5.614408,378.342749,378.342749,4.046848,7.326254,7.326254
4,0,4,0.880246,0.42378,3.415697,0.004859,0.015,8.0,6,2,...,-0.002397,0.004652,-2.933923,-1.530999,-5.034507,676.245747,676.245747,4.859057,8.302984,8.302984


In [3]:
summary = {
    'sample_count': len(df),
    'disc_count': df['disc_index'].nunique(),
    'points_per_disc_min': int(df.groupby('disc_index').size().min()),
    'points_per_disc_max': int(df.groupby('disc_index').size().max()),
    'classification_counts': df['classification'].value_counts().to_dict(),
    'first_f1_visit_fraction': float(df['dark_entry_time_s'].notna().mean()),
    'event_cap_fraction': float((df['classification'] == 'indeterminate_event_cap').mean()),
    'mean_lifetime_us': float(df['lifetime_us'].mean()),
    'median_lifetime_us': float(df['lifetime_us'].median()),
    'best_minimum_radius_mm': float(df['minimum_radius_mm'].min()),
}
summary

{'sample_count': 1250,
 'disc_count': 50,
 'points_per_disc_min': 25,
 'points_per_disc_max': 25,
 'classification_counts': {'untrapped_dark': 807,
  'indeterminate_event_cap': 440,
  'untrapped_no_reentry': 3},
 'first_f1_visit_fraction': 0.6456,
 'event_cap_fraction': 0.352,
 'mean_lifetime_us': 801.6477697571919,
 'median_lifetime_us': 808.26996161035,
 'best_minimum_radius_mm': 2.2257886924205}

## Adjustable plotting controls

In [4]:
# Adjust these values and rerun the plot cells below.
use_log_y = True
lifetime_bins = 60
f1_bins = 60
point_alpha = 0.70
point_size = 18
core_radius_mm = 2.0

In [6]:
fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)
df['classification'].value_counts().plot(kind='bar', ax=ax, color='#2563eb')
ax.set_title('Fixed-speed launch outcomes')
ax.set_xlabel('classification')
ax.set_ylabel('launches')
ax.grid(alpha=0.22, axis='y')
plt.show()

/tmp/ipykernel_76677/3083776279.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), constrained_layout=True)
axes[0].hist(df['lifetime_us'], bins=lifetime_bins, color='#2563eb', edgecolor='#0f172a', linewidth=0.35)
axes[0].set_title('Lifetime distribution')
axes[0].set_xlabel('lifetime [µs]')
axes[0].set_ylabel('launches')
finite_f1 = df['first_f1_visit_us'].dropna()
axes[1].hist(finite_f1, bins=f1_bins, color='#111827', edgecolor='#cbd5e1', linewidth=0.35)
axes[1].set_title('First F=1 visit time')
axes[1].set_xlabel('first F=1 visit [µs]')
axes[1].set_ylabel('launches')
if use_log_y:
    axes[0].set_yscale('log')
    axes[1].set_yscale('log')
for axis in axes:
    axis.grid(alpha=0.22)
plt.show()

/tmp/ipykernel_76677/911161396.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.5), constrained_layout=True)
for label, group in df.groupby('classification'):
    ax.scatter(group['s_mm'], group['lifetime_us'], s=point_size, alpha=point_alpha, label=label)
ax.set_title('Lifetime versus impact parameter')
ax.set_xlabel('impact parameter s [mm]')
ax.set_ylabel('lifetime [µs]')
ax.grid(alpha=0.22)
ax.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.5), constrained_layout=True)
for label, group in df.groupby('classification'):
    ax.scatter(group['s_mm'], group['minimum_radius_mm'], s=point_size, alpha=point_alpha, label=label)
ax.axhline(core_radius_mm, color='#15803d', linestyle='--', linewidth=1.1, label=f'{core_radius_mm:g} mm core')
ax.set_title('Closest approach versus impact parameter')
ax.set_xlabel('impact parameter s [mm]')
ax.set_ylabel('minimum radius [mm]')
ax.grid(alpha=0.22)
ax.legend()
plt.show()

In [ ]:
disc_summary = df.groupby('disc_index').agg(
    theta_rad=('theta_rad', 'first'),
    phi_rad=('phi_rad', 'first'),
    first_f1_fraction=('dark_entry_time_s', lambda x: x.notna().mean()),
    event_cap_fraction=('classification', lambda x: (x == 'indeterminate_event_cap').mean()),
    best_minimum_radius_mm=('minimum_radius_mm', 'min'),
).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), constrained_layout=True)
for ax, column, title, cmap in [
    (axes[0], 'first_f1_fraction', 'fraction with first F=1 visit', 'viridis'),
    (axes[1], 'event_cap_fraction', 'fraction hitting event cap', 'magma'),
    (axes[2], 'best_minimum_radius_mm', 'best minimum radius [mm]', 'cividis_r'),
]:
    sc = ax.scatter(disc_summary['phi_rad'], disc_summary['theta_rad'], c=disc_summary[column], s=75, cmap=cmap, edgecolor='#0f172a', linewidth=0.35)
    fig.colorbar(sc, ax=ax)
    ax.set_title(title)
    ax.set_xlabel('phi [rad]')
    ax.set_ylabel('theta [rad]')
    ax.grid(alpha=0.2)
plt.show()

In [ ]:
# Regenerate the full static analysis bundle from this notebook, if desired.
run_sampling_csv_analysis(CSV_PATH, FIGURES_DIR)